In [1]:
import pandas as pd
import numpy as np

In [2]:
# Machine Learning Algorithms
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression

# Used to split dataset into training and testing data
from sklearn.model_selection import train_test_split

# Used to evaluate classification models
from sklearn.metrics import accuracy_score, confusion_matrix

# Converts text columns into numerical values
from sklearn.preprocessing import LabelEncoder

# Part A: Dataset Understanding 

## Q1. Load the dataset and display the first five records.

In [3]:
df = pd.read_csv("Dataset 2.csv")
df.head()

,UserID,Age,Gender,SubscriptionType,WatchHoursPerWeek,DevicesUsed,FavoriteGenre,AdClicks,MonthlySpend,SubscriptionRenewed
0,1001,22,Female,Basic,23,5,Comedy,13,353,No
1,1002,55,Male,Basic,9,4,Drama,14,317,Yes
2,1003,49,Male,Basic,8,3,Comedy,16,309,No
3,1004,39,Female,Premium,19,5,Drama,45,833,Yes
4,1005,38,Female,Premium,23,5,Sci-Fi,24,804,Yes


## Q2. Determine the number of rows and columns in the dataset.

In [4]:
df.shape

(750, 10)

## Q3. Display all column names.

In [5]:
df.columns

Index(['UserID', 'Age', 'Gender', 'SubscriptionType', 'WatchHoursPerWeek',
       'DevicesUsed', 'FavoriteGenre', 'AdClicks', 'MonthlySpend',
       'SubscriptionRenewed'],
      dtype='object')

## Q4. Identify numerical and categorical features.

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750 entries, 0 to 749
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   UserID               750 non-null    int64 
 1   Age                  750 non-null    int64 
 2   Gender               750 non-null    object
 3   SubscriptionType     750 non-null    object
 4   WatchHoursPerWeek    750 non-null    int64 
 5   DevicesUsed          750 non-null    int64 
 6   FavoriteGenre        750 non-null    object
 7   AdClicks             750 non-null    int64 
 8   MonthlySpend         750 non-null    int64 
 9   SubscriptionRenewed  750 non-null    object
dtypes: int64(6), object(4)
memory usage: 58.7+ KB


## Q5. Check whether the dataset contains missing values. 

In [7]:
# This looks at each column and checks if at least one True exists. 
# If a column has even a single missing value, it returns True for that column. 
# If the column is completely clean, it returns False.
df.isna().any()

UserID                 False
Age                    False
Gender                 False
SubscriptionType       False
WatchHoursPerWeek      False
DevicesUsed            False
FavoriteGenre          False
AdClicks               False
MonthlySpend           False
SubscriptionRenewed    False
dtype: bool

# Part B: Exploratory Data Analysis 

## Q6. Calculate the average age of users. 

In [8]:
print(df["Age"].mean())

41.824


## Q7. Determine the average watch hours per week.

In [9]:
print(df["WatchHoursPerWeek"].mean())

14.236


## Q8. Find the average monthly spending of users. 

In [10]:
print(df["MonthlySpend"].mean())

689.9053333333334


## Q9. Count the number of users in each subscription category. 

In [11]:
df.groupby("SubscriptionType")["UserID"].count()

SubscriptionType
Basic      342
Premium    279
VIP        129
Name: UserID, dtype: int64

## Q10. Determine the percentage of users who renewed their subscriptions. 

In [12]:
df.groupby("SubscriptionRenewed")["UserID"].count()/len(df)*100

SubscriptionRenewed
No     53.733333
Yes    46.266667
Name: UserID, dtype: float64

# Part C: Data Preparation 

## Q11. Convert categorical features into numerical form

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750 entries, 0 to 749
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   UserID               750 non-null    int64 
 1   Age                  750 non-null    int64 
 2   Gender               750 non-null    object
 3   SubscriptionType     750 non-null    object
 4   WatchHoursPerWeek    750 non-null    int64 
 5   DevicesUsed          750 non-null    int64 
 6   FavoriteGenre        750 non-null    object
 7   AdClicks             750 non-null    int64 
 8   MonthlySpend         750 non-null    int64 
 9   SubscriptionRenewed  750 non-null    object
dtypes: int64(6), object(4)
memory usage: 58.7+ KB


In [14]:
gender_map = {"Female" : 0, "Male" : 1}
df["Gender"] = df["Gender"].map(gender_map)

subscriptionType_map = {"Basic" : 1, "Premium" : 2, "VIP": 3}
df["SubscriptionType"] = df["SubscriptionType"].map(subscriptionType_map)

favoriteGenre_map = {"Action" : 1, "Comedy" : 2, "Drama" : 3, "Horror" : 4, "Romance" : 5, "Sci-Fi" : 6}
df["FavoriteGenre"] = df["FavoriteGenre"].map(favoriteGenre_map)

subscriptionRenewed_map = {"Yes" : 1, "No" : 0}
df["SubscriptionRenewed"] = df["SubscriptionRenewed"].map(subscriptionRenewed_map)

In [15]:
df

,UserID,Age,Gender,SubscriptionType,WatchHoursPerWeek,DevicesUsed,FavoriteGenre,AdClicks,MonthlySpend,SubscriptionRenewed
0,1001,22,0,1,23,5,2,13,353,0
1,1002,55,1,1,9,4,3,14,317,1
2,1003,49,1,1,8,3,2,16,309,0
3,1004,39,0,2,19,5,3,45,833,1
4,1005,38,0,2,23,5,6,24,804,1
...,...,...,...,...,...,...,...,...,...,...
745,1746,35,1,2,13,5,1,47,732,0
746,1747,43,0,3,11,3,2,33,1388,0
747,1748,33,0,2,19,4,1,28,817,1
748,1749,33,1,1,20,4,4,26,451,1


## Q12. Define the feature set(X) and target variable (Y) for subscription renewal prediction.

In [16]:
# Define the target variable (Y)
Y = df['SubscriptionRenewed']

# Define the feature set (X) by dropping the target and the non-predictive UserID
X = df.drop(columns=['UserID', 'SubscriptionRenewed'])

# Optional: Print the shapes to verify it worked
print("X shape:", X.shape)
print("Y shape:", Y.shape)

X shape: (750, 8)
Y shape: (750,)


## Q13. Split the dataset into training and testing sets.

In [17]:
# Split the dataset into 80% training and 20% testing
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# Verify the splits by printing their shapes
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Y_train shape:", Y_train.shape)
print("Y_test shape:", Y_test.shape)

X_train shape: (600, 8)
X_test shape: (150, 8)
Y_train shape: (600,)
Y_test shape: (150,)


# Part D: Decision Tree Classification 

## Q14. Train a Decision Tree model to predict whether a user will renew their subscription.

In [19]:
# Initialize the Decision Tree Classifier
# (Using random_state=42 ensures your results are reproducible)
dt_classifier = DecisionTreeClassifier(random_state=42)

# Train (fit) the model on the training data
dt_classifier.fit(X_train, Y_train)

# Predict the subscription renewal on the test set
Y_pred_dt = dt_classifier.predict(X_test)

print("Decision Tree Model trained successfully!")

Decision Tree Model trained successfully!


## Q15. Evaluate the model using accuracy. 

In [21]:
# Calculate the accuracy score
dt_accuracy = accuracy_score(Y_test, Y_pred_dt)

# Display the accuracy as a percentage
print(f"Decision Tree Model Accuracy: {dt_accuracy * 100:.2f}%")

Decision Tree Model Accuracy: 56.00%


## Q16. Generate and interpret the confusion matrix. 

In [24]:
# Generate the raw confusion matrix array
cm = confusion_matrix(Y_test, Y_pred_dt)
print("Confusion Matrix Array:\n", cm)

Confusion Matrix Array:
 [[48 34]
 [32 36]]


# Part E : K-Nearest Neighbors (KNN) 

## Q17. Train a KNN classifier with K = 5.

In [27]:
# Initialize the KNN classifier with K = 5
knn_classifier = KNeighborsClassifier(n_neighbors=5)

# Train the model on the training data
knn_classifier.fit(X_train, Y_train)

# Predict the subscription renewal on the test set
Y_pred_knn = knn_classifier.predict(X_test)

print("KNN Model (K=5) trained successfully!")

KNN Model (K=5) trained successfully!


## Q18. Compare the accuracy of KNN with the Decision Tree model. 

In [28]:
# 1. Calculate accuracy for both models
dt_accuracy = accuracy_score(Y_test, Y_pred_dt)
knn_accuracy = accuracy_score(Y_test, Y_pred_knn)

# 2. Print the scores neatly
print("--- Accuracy Comparison ---")
print(f"Decision Tree Accuracy : {dt_accuracy * 100:.2f}%")
print(f"K-Nearest Neighbors Accuracy: {knn_accuracy * 100:.2f}%")
print("----------------------------")

# 3. Automated comparative logic statement for your report
if knn_accuracy > dt_accuracy:
    print(f"Result: KNN outperformed the Decision Tree by {(knn_accuracy - dt_accuracy)*100:.2f}%.")
elif dt_accuracy > knn_accuracy:
    print(f"Result: Decision Tree outperformed KNN by {(dt_accuracy - knn_accuracy)*100:.2f}%.")
else:
    print("Result: Both models achieved the exact same accuracy score.")

--- Accuracy Comparison ---
Decision Tree Accuracy : 56.00%
K-Nearest Neighbors Accuracy: 62.67%
----------------------------
Result: KNN outperformed the Decision Tree by 6.67%.


# Part F: Linear Regression 

## Q19. Train a Linear Regression model to predict monthly spending. 

In [34]:
# 1. Separate features and the new target variable (Monthly Spending)
X_reg = df.drop(columns=['MonthlySpend']) 
Y_reg = df['MonthlySpend']

# 2. Split into training and testing sets
X_train_reg, X_test_reg, Y_train_reg, Y_test_reg = train_test_split(
    X_reg, Y_reg, test_size=0.2, random_state=42
)

# 3. Initialize and train the Linear Regression model
lr_model = LinearRegression()
lr_model.fit(X_train_reg, Y_train_reg)

# 4. Predict on the test set
Y_pred_reg = lr_model.predict(X_test_reg)

print("Linear Regression model trained successfully!")

Linear Regression model trained successfully!


## Q20. Predict the monthly spending for a new user and interpret the result. 

In [39]:
# Predict the spending
predicted_spending = lr_model.predict([[1751, 30, 1, 1, 10, 2, 6, 0, 0]])

print(f"Predicted Monthly Spending for the new user: ${predicted_spending[0]:.2f}")

Predicted Monthly Spending for the new user: $315.07


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


# Business Reflection Questions  

## Q1. Which factors appear to influence subscription renewal the most? 

##### User engagement metrics (like watch hours per week), subscription tier type, and age appear to influence subscription renewal the most.

## Q2. Why is subscription renewal a classification problem?  

##### It is a classification problem because the outcome is categorical, predicting a distinct group choice: whether a user will renew ("Yes") or not ("No").

## Q3. Why is monthly spending a regression problem?  

##### It is a regression problem because monthly spending is a continuous, numeric value rather than a fixed label or category.

## Q4. Which algorithm performed better for renewal prediction?  

##### KNN algorithm performed better for renewal prediction due to its higher classification accuracy.

## Q5. How could the platform use these predictions to improve customer retention?

##### Netflix can use these predictions to proactively target "at-risk" users with personalized content recommendations, special offers, or discount incentives before their subscription expires.